# Part 1: Introduction

	•	Background:
    Explosion of user-generated content on platforms like Steam
    Difficulty in identifying helpful reviews
	•	Problem:
    Not all high-quality reviews receive enough exposure or votes
	•	Objective:
    Predict review helpfulness using votes_up
	•	Contribution:
    Combine text, user behavior, temporal, and game-level features
    Support better review ranking and recommendation systems

# Part 2: Dataset Description

## 2.1 Dataset 1: Steam Reviews 2020

*   Source: https://www.kaggle.com/datasets/najzeko/steam-reviews-2020
*   Dataset size: 4,374,931 reviews (22 columns).
*   Description: This dataset contains review text, timestamps, recommendation status, and community feedback metrics (e.g., votes_up, votes_funny). It also includes reviewer activity indicators. It provides primary signals for modeling review helpfulness.

## 2.2 Dataset 2: Steam Games Dataset

*   Source: https://www.kaggle.com/datasets/artermiloff/steam-games-dataset/data
*   Dataset size: 89,618 games (47 columns).
*   Description: This dataset provides game-level metadata such as release date, genre, price, and popularity indicators, enabling temporal and contextual feature engineering.

## 2.3 Data Integration

	•	Merge key: appid
	•	Purpose:
	•	Enrich review-level data with contextual game information

# Part 3: Problem Formulation

1.   Task type:

2.   Target variable:

3.   Optional transformation:

4.   Challenges:

Heavy-tailed distribution

Missing values

Multilingual text

Class imbalance (many low-vote reviews)


# Part 4: Data Preprocessing

## 4.1  Technical Preliminaries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

AttributeError: module 'numpy' has no attribute '__version__'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4.2 Data Loading

In [ ]:
# Load datasets
REVIEWS_PATH = "/content/drive/MyDrive/big_reviews.csv"
GAMES_PATH = "/content/drive/MyDrive/games_march2025_cleaned.csv"

reviews = pd.read_csv(REVIEWS_PATH)
games = pd.read_csv(GAMES_PATH)

# Basic information of datasets
print("Reviews shape:", reviews.shape)
print("Games shape:", games.shape)

print("\nReviews columns and types:")
print(reviews.info())

print("\nGames columns and types:")
print(games.info())

Reviews shape: (4374931, 22)
Games shape: (89618, 47)

Reviews columns and types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4374931 entries, 0 to 4374930
Data columns (total 22 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Unnamed: 0                   int64  
 1   appid                        int64  
 2   recommendationid             int64  
 3   language                     object 
 4   review                       object 
 5   timestamp_created            int64  
 6   timestamp_updated            int64  
 7   voted_up                     bool   
 8   votes_up                     int64  
 9   votes_funny                  int64  
 10  weighted_vote_score          float64
 11  comment_count                int64  
 12  steam_purchase               bool   
 13  received_for_free            bool   
 14  written_during_early_access  bool   
 15  steamid                      int64  
 16  num_games_owned              int64  
 17  nu

## 4.2 Dataset Overview and Column Descriptions

Before cleaning the data, let's take a closer look at the structure of the original datasets. The table below explains the meaning of each column.

In [ ]:
counts = reviews['votes_up'].value_counts().sort_index().reset_index()
counts.columns = ['votes_up', 'count']
display(counts.head(20).style.hide(axis="index"))

votes_up,count
0,3157126
1,692571
2,209699
3,87525
4,46903
5,29802
6,20606
7,15416
8,11775
9,9322



## 4.3 Data Cleaning
	•	Handle missing values
	•	Remove invalid entries
	•	Convert data types:
	•	numeric fields → pd.to_numeric
	•	timestamps → datetime

In [ ]:
# text cleanup
reviews["review"] = reviews["review"].fillna("").astype(str)
reviews["review_length"] = reviews["review"].str.len()

# timestamps cleanup
if np.issubdtype(reviews["timestamp_created"].dtype, np.number):
    reviews["review_created_dt"] = pd.to_datetime(reviews["timestamp_created"], unit="s", errors="coerce")
else:
    reviews["review_created_dt"] = pd.to_datetime(reviews["timestamp_created"], errors="coerce")

games["release_date_dt"] = pd.to_datetime(games["release_date"], errors="coerce")


## 4.3 Data Transformation
	•	Normalize numeric features
	•	Encode categorical features (One-hot encoding)
	•	Handle boolean features


## 4.4 Data Integration
	•	Merge datasets on appid
	•	Ensure consistency across datasets

In [ ]:
# Merge datasets
df = reviews.merge(games, on="appid", how="left", suffixes=("", "_game"))
print("Merged shape:", df.shape)

Merged shape: (4374931, 71)


In [ ]:
# reviewer activity fields if present
possible_numeric = [
    "num_games_owned",
    "num_reviews",
    "playtime_forever",
    "playtime_last_two_weeks",
    "playtime_at_review",
    "deck_playtime_at_review",
    "votes_funny",
    "comment_count",
    "weighted_vote_score",
    "price",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "review_score",
]

for c in possible_numeric:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# booleans -> ints
possible_bool = [
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "primarily_steam_deck"
]

for c in possible_bool:
    if c in df.columns:
        df[c] = df[c].astype("float")

# example game metadata
possible_categorical = []
for c in ["genre", "genres", "developer", "publisher", "categories", "language"]:
    if c in df.columns:
        possible_categorical.append(c)

# optional log-transform target for heavy skew
df["target_log_votes_up"] = np.log1p(df["votes_up"])

print(df[["votes_up", "target_log_votes_up", "review_length", "review_word_count", "days_since_release"]].head())

KeyError: "['review_word_count', 'days_since_release'] not in index"

# Part 5: Exploratory Data Analysis (EDA)

## 5.1 Target Distribution
	•	Histogram of votes_up
	•	Log-transformed distribution
	•	Identify skewness

## 5.2 Text Analysis
	•	Review length distribution
	•	Word count distribution
	•	Relationship with helpful votes

## 5.3 Reviewer Behavior
	•	Number of reviews per user
	•	Playtime distribution
	•	Activity vs helpfulness

## 5.4 Engagement Analysis
	•	votes_funny vs votes_up
	•	comment_count vs votes_up

## 5.5 Game-level Analysis
	•	Genre vs helpfulness
	•	Price vs helpfulness
	•	Time since release

# Part 6: Feature Engineering

## 6.1 Text Features
	•	TF-IDF representation
	•	Optional: embeddings (advanced)
	•	Length-based features:
	•	review_length
	•	word_count

## 6.2 Temporal Features
	•	Days since release
	•	Review posting time


## 6.3 Reviewer Features
	•	Activity level (num_reviews)
	•	Playtime-related features

## 6.4 Engagement Features
	•	votes_funny
	•	comment_count
	•	weighted_vote_score

## 6.5 Game Metadata
	•	Genre
	•	Price
	•	Popularity indicators

# Part 7: Modeling Approach

## 7.1 Baseline Model
	•	Linear Regression
	•	Purpose:
	•	Interpretability
	•	Benchmark performance

## 7.2 Advanced Models

# Part 8: Evaluation Strategy

## 8.1 Metrics
	•	RMSE (primary)
	•	MAE (secondary)

## 8.2 Model Comparison

# Part 9: Analysis and Discussion